# Import Libraries

In [14]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder, OrdinalEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV, StratifiedKFold, cross_validate
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier

## Model Pipelines
1. Logistic Regression
2. Random Forest
3. Gradient Boosting

In [5]:
%run /Users/festusattornelson/Documents/Projects/Python_Udemy/Projects/StudentPerformance/notebooks/02-data-preprocessing.ipynb


Training Samples: 8000
Testing Samples: 2000

Numerical Columns:
['study_hours', 'attendance', 'sleep_hours', 'internet_usage', 'assignments_completed', 'previous_score', 'exam_score']

Categorical Columns:
[]


In [6]:
models = {

    "Logistic Regression": Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=2000
        ))
    ]),

    "Random Forest": Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            random_state=42
        ))
    ]),

    "Gradient Boosting": Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", GradientBoostingClassifier(
            random_state=42
        ))
    ])
}

## Grid search parameters

In [7]:
param_grids = {

    "Logistic Regression": {
        "classifier__C": [0.01, 0.1, 1, 10, 100]
    },

    "Random Forest": {
        "classifier__n_estimators": [100, 200, 300],
        "classifier__max_depth": [None, 5, 10, 20],
        "classifier__min_samples_split": [2, 5, 10]
    },

    "Gradient Boosting": {
        "classifier__n_estimators": [100, 200],
        "classifier__learning_rate": [0.01, 0.05, 0.1],
        "classifier__max_depth": [3, 5]
    }
}

## Crosss validation setup

In [11]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision_weighted",
    "recall": "recall_weighted",
    "f1": "f1_weighted"
}

## Training loop and Hyperparameter Tuning

In [12]:
results = []

best_model = None
best_model_name = None
best_score = -1

for name, pipeline in models.items():

    print(f"MODEL: {name}")
    print("="*70)

    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grids[name],
        scoring="f1_weighted",
        cv=cv,
        n_jobs=-1,
        verbose=1
    )

    grid.fit(X_train, y_train)

    tuned_model = grid.best_estimator_

    print("\nBest Parameters:")
    print(grid.best_params_)

MODEL: Logistic Regression
Fitting 5 folds for each of 5 candidates, totalling 25 fits

Best Parameters:
{'classifier__C': 100}
MODEL: Random Forest
Fitting 5 folds for each of 36 candidates, totalling 180 fits

Best Parameters:
{'classifier__max_depth': None, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 100}
MODEL: Gradient Boosting
Fitting 5 folds for each of 12 candidates, totalling 60 fits

Best Parameters:
{'classifier__learning_rate': 0.01, 'classifier__max_depth': 3, 'classifier__n_estimators': 100}


## Cross validation metrics

In [15]:
cv_scores = cross_validate(
        tuned_model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )
